# Geografía de presentación

Proyección y simplificación visual de polígonos oficiales y conexiones observadas. Las coordenadas y geometrías de análisis se conservan en sus tablas y fuentes. El SVG se utiliza como una representación local que responde a los filtros del reporte.

In [ ]:
import math
import numpy as np
import pandas as pd

def simplificar_linea(puntos, tolerancia):
    if len(puntos) < 3: return puntos
    a, b = puntos[0], puntos[-1]
    vector = b-a; norma = float(vector@vector)
    if norma == 0: distancias=np.linalg.norm(puntos-a,axis=1)
    else:
        t=np.clip((puntos-a)@vector/norma,0,1)
        distancias=np.linalg.norm(puntos-(a+t[:,None]*vector),axis=1)
    indice=int(distancias.argmax())
    if distancias[indice] <= tolerancia: return puntos[[0,-1]]
    return np.vstack([simplificar_linea(puntos[:indice+1],tolerancia)[:-1],
                      simplificar_linea(puntos[indice:],tolerancia)])


In [ ]:
def preparar_geografia_svg(puntos, geografia):
    anillos=[]
    for feature in geografia["features"]:
        g=feature["geometry"]
        poligonos=[g["coordinates"]] if g["type"]=="Polygon" else g["coordinates"]
        anillos.extend(np.asarray(r,dtype=float)[:,:2] for p in poligonos for r in p)
    coordenadas=np.vstack(anillos+[puntos[["Longitud","Latitud"]].astype(float).to_numpy()])
    assert np.isfinite(coordenadas).all() and (np.abs(coordenadas[:,1])<85).all()
    def mercator(v): return np.column_stack([v[:,0],np.degrees(np.log(np.tan(np.pi/4+np.radians(v[:,1])/2)))])
    plano=mercator(coordenadas); minimo=plano.min(axis=0); maximo=plano.max(axis=0)
    escala=min(710/(maximo[0]-minimo[0]),540/(maximo[1]-minimo[1]))
    centro=(minimo+maximo)/2
    def proyectar(v):
        a=(mercator(v)-centro)*escala
        return np.column_stack([a[:,0]+385,300-a[:,1]])
    trazos=[]
    for tolerancia in (.25,.5,1,1.5,2,3):
        trazos=[]
        for anillo in anillos:
            v=simplificar_linea(proyectar(anillo),tolerancia)
            trazos.append("M"+"L".join(f"{x:.1f} {y:.1f}" for x,y in v)+"Z")
        if sum(map(len,trazos)) < 11000: break
    return proyectar, trazos


In [ ]:
def enriquecer_mapa_local(puntos, geografia):
    base=puntos.drop(columns=["TrazoSVG"],errors="ignore").copy()
    proyectar,anillos=preparar_geografia_svg(base,geografia)
    xy=proyectar(base[["Longitud","Latitud"]].astype(float).to_numpy())
    posiciones=pd.DataFrame({"RutaKey":base.RutaKey.to_numpy(),"Orden":base.Orden.astype(int).to_numpy(),
        "X":xy[:,0],"Y":xy[:,1]})
    origen=posiciones.loc[posiciones.Orden.eq(0)].set_index("RutaKey")
    destino=posiciones.loc[posiciones.Orden.eq(1)].set_index("RutaKey")
    assert origen.index.is_unique and destino.index.is_unique and set(origen.index)==set(destino.index)
    volumen=base.loc[base.Orden.astype(int).eq(0)].copy()
    maximos=volumen.groupby("Ano").Viajes.transform("max").astype(float)
    anchos=1+3*np.sqrt(volumen.Viajes.astype(float).to_numpy()/maximos.to_numpy())
    rutas={}
    for clave,ancho in zip(volumen.RutaKey,anchos):
        a,b=origen.loc[clave],destino.loc[clave]
        rutas[clave]=f"<path d='M{a.X:.1f} {a.Y:.1f}L{b.X:.1f} {b.Y:.1f}' stroke-width='{ancho:.1f}'/>"
    base["TrazoSVG"]=""
    seleccion=base.Orden.astype(int).eq(0)
    base.loc[seleccion,"TrazoSVG"]=base.loc[seleccion,"RutaKey"].map(rutas)
    barrios="".join(f"<path d='{d}'/>" for d in anillos)
    svg=("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 1000 600'>"
         "<g fill='#F3F7FC' stroke='#B8CBDC' stroke-width='.7'>"+barrios+"</g></svg>")
    assert len(svg)<32000 and max(map(len,rutas.values()))<32000
    assert len(base)==len(puntos) and base.loc[seleccion,"TrazoSVG"].notna().all()
    return base,svg
